# 正向合成轨迹可视化（forward_trajectories）

- 数据源：`../data/forward_trajectories.csv`
- 选择：先选 `ligand_smiles`，再选 `route_id`；仅在选择具体路径时渲染。
- 展示：按 `step_index` 升序展示每一步的 state + building_block → result，并标注反应模板；显示候选数量统计。


In [ ]:
import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D

CSV_PATH = "../data/forward_trajectories.csv"
assert os.path.exists(CSV_PATH), f"CSV 文件未找到：{CSV_PATH}"

# 读取数据
usecols = [
    "ligand_smiles",
    "route_id",
    "step_index",
    "route_score",
    "state_smiles",
    "action_building_block",
    "action_reaction_template",
    "result_smiles",
    "is_start_state",
    "num_intermediate_candidates",
    "num_building_blocks",
    "forward_step_index",
    "is_in_forward_chain",
]

df = pd.read_csv(CSV_PATH, usecols=usecols)

def _to_bool(x):
    if isinstance(x, bool):
        return x
    s = str(x).strip().lower()
    return s in {"true", "1", "t", "yes", "y"}

if "is_start_state" in df.columns:
    df["is_start_state"] = df["is_start_state"].apply(_to_bool)

# 组件
ligands = sorted(df["ligand_smiles"].dropna().unique().tolist())
ligand_dropdown = widgets.Dropdown(
    options=ligands,
    value=ligands[0] if ligands else None,
    description='Ligand:',
    layout=widgets.Layout(width='40%'),
)

route_dropdown = widgets.Dropdown(
    options=[("请选择路径", None)],
    value=None,
    description='路径ID:',
    layout=widgets.Layout(width='40%'),
)

view_mode = widgets.ToggleButtons(
    options=[("主链", "chain"), ("全部", "all")],
    value="chain",
    description='显示:',
    button_style='',
    layout=widgets.Layout(width='20%'),
)

header_html = widgets.HTML(value="")
output = widgets.Output()


def to_svg(smiles: str, w=220, h=220):
    try:
        mol = Chem.MolFromSmiles(smiles)
    except Exception:
        mol = None
    if mol is None:
        return "<div style='color:red; border:1px solid red; padding:8px;'>Invalid SMILES</div>"
    d = rdMolDraw2D.MolDraw2DSVG(w, h)
    rdMolDraw2D.PrepareAndDrawMolecule(d, mol)
    d.FinishDrawing()
    return d.GetDrawingText()


def _is_na_like(x):
    if x is None:
        return True
    s = str(x).strip().lower()
    return s in {"", "na", "n/a", "nan", "none", "null"}


def render_step_html(row):
    state_svg = to_svg(str(row.get("state_smiles", "")), w=180, h=180)
    result_svg = to_svg(str(row.get("result_smiles", "")), w=180, h=180)

    block_raw = row.get("action_building_block", "")
    block = None if _is_na_like(block_raw) else str(block_raw)
    block_svg = to_svg(block, w=160, h=160) if block else "<div style='color:#888;'>—</div>"

    tpl = str(row.get("action_reaction_template", ""))
    step_idx = row.get("step_index")

    nic = row.get("num_intermediate_candidates")
    nbb = row.get("num_building_blocks")
    nic_txt = "N/A" if _is_na_like(nic) else str(int(nic))
    nbb_txt = "N/A" if _is_na_like(nbb) else str(int(nbb))

    html = f"""
    <div style='margin:12px 0; padding:10px; border:1px solid #ddd; border-radius:8px;'>
      <div style='margin-bottom:8px; color:#555;'>Step {step_idx} | 反应模板: {tpl} | 候选中间体数: {nic_txt} | 备选砌块数: {nbb_txt}</div>
      <div style='display:flex; align-items:center; gap:16px;'>
        <div style='text-align:center;'>
          <div style='margin-bottom:6px; font-weight:bold;'>State</div>
          {state_svg}
        </div>
        <div style='font-size: 28px; padding: 0 4px;'>+</div>
        <div style='text-align:center;'>
          <div style='margin-bottom:6px; font-weight:bold;'>Building block</div>
          {block_svg}
        </div>
        <div style='font-size: 36px; padding: 0 8px;'>→</div>
        <div style='text-align:center;'>
          <div style='margin-bottom:6px; font-weight:bold;'>Result</div>
          {result_svg}
        </div>
      </div>
    </div>
    """
    return html


def _prefer_non_empty_block(rows):
    rows = list(rows)
    if not rows:
        return None
    # Prefer rows with non-empty building block, then smallest step_index
    rows.sort(key=lambda r: (_is_na_like(r.get("action_building_block")), r.get("step_index") if r.get("step_index") is not None else 10**9))
    return rows[0]


def order_route_rows(sub_df: pd.DataFrame) -> pd.DataFrame:
    if sub_df.empty:
        return sub_df
    # Work on a copy to avoid mutating the global df
    g = sub_df.copy()

    # Build helper maps
    result_to_rows = {}
    state_to_rows = {}
    for idx, r in g.iterrows():
        rs = str(r.get("result_smiles", ""))
        ss = str(r.get("state_smiles", ""))
        result_to_rows.setdefault(rs, []).append(r)
        state_to_rows.setdefault(ss, []).append(r)

    # Candidates to start: is_start_state or state not produced by any result (or empty)
    start_candidates = []
    for _, r in g.iterrows():
        ss = str(r.get("state_smiles", ""))
        if bool(r.get("is_start_state")) or _is_na_like(ss) or ss not in result_to_rows:
            start_candidates.append(r)

    def build_chain(start_row):
        chain = [start_row]
        seen_pairs = {(str(start_row.get("state_smiles", "")), str(start_row.get("result_smiles", "")), str(start_row.get("action_building_block", "")), str(start_row.get("action_reaction_template", "")))}
        current = start_row
        # Greedily follow: next rows whose state equals current result
        while True:
            curr_res = str(current.get("result_smiles", ""))
            candidates = [r for r in g.itertuples(index=False) if str(getattr(r, "state_smiles", "")) == curr_res]
            if not candidates:
                break
            # Convert back to Series-like dicts
            candidates_dicts = [g.iloc[i]._asdict() if hasattr(g.iloc[i], "_asdict") else g.iloc[i].to_dict() for i in []]  # placeholder to keep linter happy
            # Use DataFrame rows directly
            candidates_rows = g[g["state_smiles"].astype(str) == curr_res].to_dict("records")
            if not candidates_rows:
                break
            # Prefer with non-empty building block; tie-break by step_index
            next_row = _prefer_non_empty_block(candidates_rows)
            pair = (str(next_row.get("state_smiles", "")), str(next_row.get("result_smiles", "")), str(next_row.get("action_building_block", "")), str(next_row.get("action_reaction_template", "")))
            if pair in seen_pairs:
                break
            seen_pairs.add(pair)
            chain.append(next_row)
            current = next_row
        return chain

    # Try all starts and keep the longest chain
    best_chain = []
    # Sort starts: prefer is_start_state then smallest step_index
    start_candidates.sort(key=lambda r: (0 if bool(r.get("is_start_state")) else 1, r.get("step_index") if r.get("step_index") is not None else 10**9))
    for sr in start_candidates:
        c = build_chain(sr)
        if len(c) > len(best_chain):
            best_chain = c

    if not best_chain:
        # Fallback: sort by step_index
        return g.sort_values(by=["step_index"])  

    # Build ordered DataFrame matching the chain
    ordered = pd.DataFrame(best_chain)
    # Preserve original columns order when possible
    ordered = ordered[[c for c in g.columns if c in ordered.columns]]
    return ordered


def render_route(lig_smi, route_id):
    sub = df[(df["ligand_smiles"] == lig_smi) & (df["route_id"] == route_id)]
    if sub.empty:
        return "<div style='color:red;'>该路径无数据</div>"

    mode = "chain"
    try:
        mode = view_mode.value
    except Exception:
        pass

    if mode == "chain":
        if "forward_step_index" in sub.columns and sub["forward_step_index"].notna().any():
            ordered = sub[sub.get("is_in_forward_chain", True)].sort_values(by=["forward_step_index"]).copy()
            ordering_note = "主链 | 按 forward_step_index 渲染"
        else:
            ordered = order_route_rows(sub)
            ordering_note = "主链 | 按 result=next.state 贪心重建"
    else:
        # 全部条目视图：尽量把主链排在前面
        if "forward_step_index" in sub.columns and sub["forward_step_index"].notna().any():
            ordered = sub.sort_values(
                by=["is_in_forward_chain", "forward_step_index", "step_index"],
                ascending=[False, True, True],
            ).copy()
            ordering_note = "全部 | 先主链，再按 forward_step_index/step_index 排序"
        else:
            ordered = sub.sort_values(by=["step_index"]).copy()
            ordering_note = "全部 | 无 forward_step_index，按 step_index 排序"

    steps = "\n".join([render_step_html(r) for _, r in ordered.iterrows()])
    score = sub["route_score"].dropna().unique()
    score_text = ", ".join(map(str, score)) if len(score) else "N/A"
    return f"""
    <div>
      <div style='margin:6px 0; color:#444;'>路线分数: {score_text} | 步骤数(展示): {len(ordered)} / 全部条目: {len(sub)} | {ordering_note}</div>
      {steps}
    </div>
    """


def render_ligand_header(lig_smi):
    sub = df[df["ligand_smiles"] == lig_smi]
    if sub.empty:
        return "<div style='color:red;'>该 ligand 无轨迹</div>"
    route_ids = sub["route_id"].dropna().unique().tolist()
    header_svg = to_svg(lig_smi, w=180, h=180)
    return f"""
    <div>
      <div style='display:flex; align-items:center; gap:12px;'>
        <div style='font-weight:bold;'>Ligand:</div>
        {header_svg}
      </div>
      <div style='margin:6px 0; color:#444;'>共 {len(route_ids)} 条路径。请选择路径进行渲染。</div>
    </div>
    """


def on_ligand_change(change):
    lig = change["new"]
    with output:
        output.clear_output(wait=True)
    if lig is None:
        route_dropdown.options = [("请选择路径", None)]
        route_dropdown.value = None
        header_html.value = ""
        return
    sub = df[df["ligand_smiles"] == lig]
    route_ids = sub["route_id"].dropna().unique().tolist()
    route_dropdown.options = [("请选择路径", None)] + [(rid, rid) for rid in route_ids]
    route_dropdown.value = None
    header_html.value = render_ligand_header(lig)


def on_route_change(change):
    rid = change["new"]
    with output:
        output.clear_output(wait=True)
    if rid is None or ligand_dropdown.value is None:
        return
    html = render_route(ligand_dropdown.value, rid)
    display(HTML(html))


def on_view_mode_change(change):
    rid = route_dropdown.value
    if rid is None or ligand_dropdown.value is None:
        return
    with output:
        output.clear_output(wait=True)
    html = render_route(ligand_dropdown.value, rid)
    display(HTML(html))


ligand_dropdown.observe(on_ligand_change, names='value')
route_dropdown.observe(on_route_change, names='value')
view_mode.observe(on_view_mode_change, names='value')

display(HTML("<h2>正向合成轨迹可视化</h2>"))
widgets.VBox([
    widgets.HBox([ligand_dropdown, route_dropdown, view_mode]),
    header_html,
    output
])


In [2]:
# 初始化：仅更新路径选择，不自动渲染
on_ligand_change({"new": ligand_dropdown.value})
print("界面已准备，选择 ligand 与 路径后渲染")


界面已准备，选择 ligand 与 路径后渲染
